# 04 - ALU: The Brain of the CPU

## Introduction

The **Arithmetic Logic Unit (ALU)** is the computational core of every CPU. It takes two 8-bit inputs and an operation code, performs the operation, and reports **status flags** describing the result.

We'll implement an ALU that supports:
- **Arithmetic**: ADD, SUB
- **Logic**: AND, OR, XOR, NOT
- **Shift**: SHL, SHR
- **Compare**: CMP

## Learning Objectives

1. Understand ALU architecture and opcode dispatch
2. Implement multiple operations built from your gates and adders
3. Understand the four status flags (Z, C, N, V) - they power the conditional jumps you'll build later

In [ ]:
# Auto-reload: your edits to src/computer/*.py are picked up on each cell run
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

from computer import int_to_bits, bits_to_int

print("Setup complete!")

## ALU Architecture

```mermaid
flowchart TB
    A["A[7:0]"] --> ALU[ALU]
    B["B[7:0]"] --> ALU
    Op["Opcode[3:0]"] --> ALU
    ALU --> Result["Result[7:0]"]
    ALU --> Flags["Flags (Z,C,N,V)"]
```

## Opcode Table

| # | Operation | Description | As LSB-first bits |
|---|-----------|-------------|-------------------|
| 0 | ADD | A + B | `[0, 0, 0, 0]` |
| 1 | SUB | A - B | `[1, 0, 0, 0]` |
| 2 | AND | A & B | `[0, 1, 0, 0]` |
| 3 | OR | A \| B | `[1, 1, 0, 0]` |
| 4 | XOR | A ^ B | `[0, 0, 1, 0]` |
| 5 | NOT | ~A | `[1, 0, 1, 0]` |
| 6 | SHL | A << 1 | `[0, 1, 1, 0]` |
| 7 | SHR | A >> 1 | `[1, 1, 1, 0]` |
| 8 | CMP | Compare A, B | `[0, 0, 0, 1]` |

> **⚠️ LSB-first reminder:** throughout this project, bit lists put the **least significant bit at index 0**. Operation number 1 (SUB) is the list `[1, 0, 0, 0]` - the 1 is at index 0 because that's the 1s place. Written as a conventional binary string (MSB first) it would be `0001`. The `ALU.OP_*` class constants spare you from ever typing these by hand.

## Status Flags

The ALU doesn't just produce a result - it also reports four **flags** that describe the result. These flags are how programs make decisions: `JZ`/`JNZ` (notebook 12) jump or not based on the Z flag that the ALU set here.

| Flag | Name | Set to 1 when... |
|------|------|------------------|
| **Z** | Zero | the result is exactly 0 |
| **C** | Carry / Borrow | ADD overflowed past 255 (carry-out) · SUB/CMP needed a borrow (B > A, unsigned) · SHL shifted out the old MSB · SHR shifted out the old LSB |
| **N** | Negative | the result's MSB (bit 7) is 1 - in two's complement, that means negative |
| **V** | Overflow | the *signed* result is wrong: adding two same-sign numbers produced the opposite sign |

Worked examples:

- `255 + 1` → result `0`, **Z=1, C=1** (the carry left the 8-bit world)
- `3 - 5` → result `254` (that's -2 in two's complement), **C=1** (borrow), **N=1** (MSB set)
- `100 + 100` → result `200`, **N=1, V=1** (two positives "summed" to a negative - signed overflow)

## CMP: Compare Without Changing Anything

`CMP` performs `A - B` **only for the flags**: it computes the subtraction, sets Z/C/N/V from it, but returns `A` unchanged. That's how real CPUs implement comparisons - `A == B` becomes "subtract and check Z".

**✏️ Your turn:** open [`src/computer/alu.py`](../src/computer/alu.py) and implement the `ALU` class: the small per-operation helpers (`_add`, `_and`, `_shl`, ...), `_calculate_flags`, and the opcode dispatch in `__call__` - each docstring is the full spec.

Then run the cell below. Thanks to `%autoreload`, every time you save the file and re-run a cell, your latest code is used.

In [ ]:
# Exercise: implement the ALU in src/computer/alu.py, then run this cell.
from computer.alu import ALU

alu = ALU()


def show(label, a, b, op):
    """Run one ALU operation and print result + flags."""
    output = alu(int_to_bits(a, 8), int_to_bits(b, 8), op)
    if output is None:
        print(f"{label:14} -> None (not implemented yet)")
        return
    result, flags = output
    print(f"{label:14} -> {bits_to_int(result):3d}   flags={flags}")


show("ADD 5, 3", 5, 3, ALU.OP_ADD)
show("ADD 255, 1", 255, 1, ALU.OP_ADD)   # expect Z=1, C=1
show("SUB 3, 5", 3, 5, ALU.OP_SUB)       # expect 254, C=1 (borrow), N=1
show("AND 0xF0, 0xAA", 0xF0, 0xAA, ALU.OP_AND)
show("XOR 0x55, 0xAA", 0x55, 0xAA, ALU.OP_XOR)
show("NOT 0x0F", 0x0F, 0, ALU.OP_NOT)
show("SHL 0x81", 0x81, 0, ALU.OP_SHL)    # expect 0x02, C=1 (old MSB shifted out)
show("CMP 5, 5", 5, 5, ALU.OP_CMP)       # expect result 5 (unchanged), Z=1

## Validation

In [ ]:
from utils.checker import check

check("alu")

## Summary

The ALU is complete! It can:
- Perform 9 different operations, all built from your gates and adders
- Set the Z, C, N, V status flags that conditional branching will rely on
- Form the computational core of our CPU

### What's Next?

We need to store data! In the next notebooks, we'll build:
- Latches and Flip-Flops (memory elements)
- Registers (groups of flip-flops)
- Counters (for the program counter)
- RAM (main memory)